## Spectabular Library

**Emil Sekerinski, McMaster University, April 2026**

This Jupyter notebook is the `Spectabular` tool that accompanies the FormaliSE 2026 paper *A Spectabular Model of an Automotive Adaptive Exterior Light System*. Spectabular is a Python library that requires the Z3 solver, https://pypi.org/project/z3-solver. Spectabular must be imported into a notebook with:

```
%run "spectabular.ipynb"
```

This is a development version with embedded tests. To run the test, set `TEST` to `True`. Do not import Spectabular with `TEST` set to `True`, as that will run all tests and pollute the namespace. 

In [1]:
TEST = False

In [2]:
from z3 import And, Or, Not, Implies, If, Sum, Exists, ForAll, z3

### Basic Types

Spectabular supports Boolean, integer, and real variables, vectors of those, and enumerations. Each declaration introduces an unprimed and a primed variable:

In [3]:
def Bool(n: str):
    globals()[n] = z3.Bool(n)
    globals()[n + 'ʹ'] = z3.Bool(n + 'ʹ')

def Int(n: str):
    globals()[n] = z3.Int(n)
    globals()[n + 'ʹ'] = z3.Int(n + 'ʹ')

def Real(n: str):
    globals()[n] = z3.Real(n)
    globals()[n + 'ʹ'] = z3.Real(n + 'ʹ')

For test cases below, we use:

In [4]:
if TEST:
    Bool('a'); Bool('b'); Bool('c'); Bool('d'); Bool('e')
    Int('x'); Int('y'); Int('z')

Procedure `EnumType` declares an enumeration type; procedure `Enum` declares an enumeration variable:

In [5]:
def EnumType(t: str, *vs: tuple[str]): # type t, tuple vs of values
    es = z3.EnumSort(t, vs)
    globals()[t + '_z3'] = es[0] # t_z3 is the Z3 sort
    exec("def " + t + "(v): globals()[v] = z3.Const(v, " + t + "_z3); globals()[v + 'ʹ'] = z3.Const(v + 'ʹ', " + t + "_z3)",
         globals())
    for n, v in zip(vs, es[1]): globals()[n] = v

In [6]:
def Enum(n: str, *vs: tuple[str]): # name n, tuple vs of values
    es = z3.EnumSort(n, vs)
    globals()[n] = z3.Const(n, es[0])
    globals()[n + 'ʹ'] = z3.Const(n + 'ʹ', es[0])
    for e, v in zip(vs, es[1]): globals()[e] = v

An enumeration type `E` is the Z3 sort `E_z3`. For example, the following prints `Color`:

In [7]:
if TEST: EnumType('Color', 'Red', 'Green', 'Blue'); print(Color_z3, type(Color_z3))

Declaring an enumeration type `E` creates a function `E` that allows variables of that type to be declared. The following declares variable `col` of type `Color`; variables of an enumeration type are of Z3 type `DatatypeRef`:

In [8]:
if TEST: Color('col'); print(Color, col, colʹ, type(col))

In [9]:
def BoolVector(n: str, l: int):
    globals()[n] = z3.BoolVector(n, l)
    globals()[n + 'ʹ'] = z3.BoolVector(n + 'ʹ', l)

def IntVector(n: str, l: int):
    globals()[n] = z3.IntVector(n, l)
    globals()[n + 'ʹ'] = z3.IntVector(n + 'ʹ', l)

def RealVector(n: str, l: int):
    globals()[n] = z3.RealVector(n, l)
    globals()[n + 'ʹ'] = z3.RealVector(n + 'ʹ', l)

Auxiliary function `pyToZ3` converts Python Boolean, integer, and float constants to Z3 constants for constructing tables. If the argument is a tuple or list, the conversion is performed recursively, preserving the structure. If the argument is none of these the argument is returned unchanged:

In [10]:
def pyToZ3(x):
    return z3.BoolVal(x) if type(x) == bool else \
           z3.IntVal(x) if type(x) == int else \
           z3.RealVal(x) if type(x) == float else \
           tuple(pyToZ3(e) for e in x) if isinstance(x, (tuple, list)) else x

The following prints `7` of type `z3.IntNumRef`:

In [11]:
if TEST: assert type(pyToZ3(7)) == z3.IntNumRef; print(pyToZ3(7))

#### Totality and Disjointness

Vector `pv` is total if (at least) one element is true:

    total(pv) ≡ ⋁ 𝑖 • pvᵢ

Totality is determined by checking if the negation is unsatisfiable; if it is satisfiable, then the model is a counterexample for totality. 

In [12]:
def checktotal(pv: tuple[z3.BoolRef, ...]) -> tuple[bool|None, z3.ModelRef|None]:
    s = z3.Solver()
    s.add(~Or(pv))
    v = s.check()
    return (True, None) if v == z3.unsat else \
        (False, s.model()) if v == z3.sat else (None, None)

Vector `pv` is _disjoint_ if any two distinct elements are mutually exclusive:

    disjoint(pv) ≡ ⋀ i, k | i ≠ k · ¬(pvᵢ ∧ pvₖ)

Disjointness is determined by checking if the negation is unsatisfiable; if it is satisfiable, then the model is a counterexample for disjointness. The implementation checks each pair, stopping when a satisfiable pair is found.

In [13]:
def checkdisjoint(pv: tuple[z3.z3.BoolRef, ...]) -> tuple[bool|None, z3.z3.ModelRef|None]:
    for i in range(len(pv)):
        for k in range(i + 1, len(pv)):
            s = z3.Solver()
            s.add(pv[i] & pv[k])
            v = s.check()
            if v == z3.sat: return (False, s.model())
            elif v != z3.unsat: return (None, None)
    return (True, None)

#### Specification Classes

The goal is to
- define predicate and relation tables as a new type (class),
- allow all logical operators (`¬`, `∧`, `∨`, `⇒`, `⇐`, `=`, `≠`) on flat expressions and tables
- define domain (`∆`), possibility (`⟨_⟩_`), necessity (`[_]_`), and correctness (`{_}_{_}`) for use with plain expressions and tables.

The design is:
- Class `SpecExpr` is an abstract superclass: operators `~`, `&`, `|`, `>>`, `<<`, `==`, `!=` call constructors of subclasses; method `_repr_html_` wraps the output of method `mathml` into html so it can be displayed.
- Classes `NotSpec`, `AndSpec`, ... are subclasses and serve as constructors for specification expressions; all have auxiliary method `mathml` for printing the expression.
- Class `Table` is also a subclass and is used for constructing tables.
- Expressions are either of type `SpecExpr` or Z3 expressions of type `z3.ExprRef`.

In [14]:
# class Spec:

# from mathml_to_latex.converter import MathMLToLaTeX

# mathml = """
# <math>
#     <mrow>
#         <mi>A</mi>
#         <mo>=</mo>
#         <mfenced open = "[" close="]">
#         <mtable>
#             <mtr>
#             <mtd><mi>x</mi></mtd>
#             <mtd><mi>y</mi></mtd>
#             </mtr>
#             <mtr>
#             <mtd><mi>z</mi></mtd>
#             <mtd><mi>w</mi></mtd>
#             </mtr>
#         </mtable>
#         </mfenced>
#     </mrow>
# </math>
# """

# result = MathMLToLaTeX().convert(mathml)
# # A = \begin{bmatrix} x & y \\ z & w \end{bmatrix}

# Running Tests

In [15]:
from mathml_to_latex.converter import MathMLToLaTeX

type Expr = z3.ExprRef | SpecExpr

class SpecExpr:
    def __invert__(self):
        return NotExpr(self)
    def __and__(self, other: Expr):
        return AndExpr(self, other)
    def __rand__(self, other: Expr): # TODO
        return AndExpr(self, other)
    def __or__(self, other: Expr):
        return OrExpr(self, other)
    def __rshift__(self, other: Expr):
        return ImpliesExpr(self, other)
    def __lshift__(self, other: Expr):
        return ConsExpr(self, other)
    def __eq__(self, other: Expr):
        return EqExpr(self, other)
    def __ne__(self, other: Expr):
        return NeExpr(self, other)
    def _repr_html_(self):
        return '<math style="border-left:2em solid white;font-family: sans-serif;">' + self.mathml() + '</math>'

ModuleNotFoundError: No module named 'mathml_to_latex'

In [ ]:
# TODO
# Bool('a'); Int('x'); Int('y')
# b = True
# b & Table((a, ~a), (x + y >= 3, True))

In [ ]:
class NotExpr(SpecExpr):
    def __init__(self, t: Expr):
        self.arg = t
    def mathml(self) -> str:
        return '<mrow>¬<mo lspace=".1em">(</mo>' + self.arg.mathml() + '<mo>)</mo></mrow>'

In [ ]:
class AndExpr(SpecExpr):
    def __init__(self, t0: Expr, t1: Expr):
        self.arg0, self.arg1 = t0, t1
    def mathml(self) -> str:
        return '<mrow><mo>(</mo>' + self.arg0.mathml() + '<mo>)</mo></mrow><mo lspace=".1em" rspace=".1em">∧</mo><mrow><mo>(</mo>' + \
            self.arg1.mathml() + '<mo>)</mo></mrow>'

In [ ]:
class OrExpr(SpecExpr):
    def __init__(self, t0: Expr, t1: Expr):
        self.arg0, self.arg1 = t0, t1
    def mathml(self) -> str:
        return '<mrow><mo>(</mo>' + self.arg0.mathml() + '<mo>)</mo></mrow><mo lspace=".1em" rspace=".1em">∨</mo><mrow><mo>(</mo>' + \
            self.arg1.mathml() + '<mo>)</mo></mrow>'

In [ ]:
class ImpliesExpr(SpecExpr):
    def __init__(self, t0: Expr, t1: Expr):
        self.arg0, self.arg1 = t0, t1
    def mathml(self) -> str:
        return '<mrow><mo>(</mo>' + self.arg0.mathml() + '<mo>)</mo></mrow><mo lspace=".1em" rspace=".1em">⇒</mo><mrow><mo>(</mo>' + \
            self.arg1.mathml() + '<mo>)</mo></mrow>'

In [ ]:
class ConsExpr(SpecExpr):
    def __init__(self, t0: Expr, t1: Expr):
        self.arg0, self.arg1 = t0, t1
    def mathml(self) -> str:
        return '<mrow><mo>(</mo>' + self.arg0.mathml() + '<mo>)</mo></mrow><mo lspace=".1em" rspace=".1em">⇐</mo><mrow><mo>(</mo>' + \
            self.arg1.mathml() + '<mo>)</mo></mrow>'

In [ ]:
class EqExpr(SpecExpr):
    def __init__(self, t0: Expr, t1: Expr):
        self.arg0, self.arg1 = t0, t1
    def mathml(self) -> str:
        return '<mrow><mo>(</mo>' + self.arg0.mathml() + '<mo>)</mo></mrow><mo lspace=".1em" rspace=".1em">=</mo><mrow><mo>(</mo>' + \
            self.arg1.mathml() + '<mo>)</mo></mrow>'

In [ ]:
class NeExpr(SpecExpr):
    def __init__(self, t0: Expr, t1: Expr):
        self.arg0, self.arg1 = t0, t1
    def mathml(self) -> str:
        return '<mrow><mo>(</mo>' + self.arg0.mathml() + '<mo>)</mo></mrow><mo lspace=".1em" rspace=".1em">≠</mo><mrow><mo>(</mo>' + \
            self.arg1.mathml() + '<mo>)</mo></mrow>'

In [ ]:
class Dom(SpecExpr):
    def __init__(self, t: Expr):
        self.arg = t
    def mathml(self) -> str:
        return '<mrow>∆<mo lspace=".1em";>(</mo>' + self.arg.mathml() + '<mo>)</mo></mrow>'

In [ ]:
class Possible(SpecExpr):
    def __init__(self, t0: Expr, t1: Expr):
        self.arg0, self.arg1 = t0, t1
    def mathml(self) -> str:
        return '<mrow><mo>⟨</mo>' + self.arg0.mathml() + '<mo rspace=".2em">⟩</mo>' + \
            self.arg1.mathml() + '</mrow>'

In [ ]:
class Necessary(SpecExpr):
    def __init__(self, t0: Expr, t1: Expr):
        self.arg0, self.arg1 = t0, t1
    def mathml(self) -> str:
        return '<mrow><mo>[</mo>' + self.arg0.mathml() + '<mo rspace=".2em">]</mo>' + \
            self.arg1.mathml() + '</mrow>'

In [ ]:
class Correct(SpecExpr):
    def __init__(self, t0: Expr, t1: Expr, t2: Expr):
        self.arg0, self.arg1, self.arg2 = t0, t1, t2
    def mathml(self) -> str:
        return '<mrow>' + (self.arg0 if type(self.arg0) == str else self.arg0.mathml()) + \
            '<mo lspace=".2em">{</mo>' + (self.arg1 if type(self.arg1) == str else self.arg1.mathml()) + \
            '<mo rspace=".2em">}</mo>' + (self.arg2 if type(self.arg2) == str else self.arg2.mathml()) + '</mrow>'

In [ ]:
class Table(SpecExpr):
    def __init__(self, *table, top = None, left = None, body = None):
        if body == None: # table must be well-formed
            assert top == None and left == None, 'inconsistent table specification'
            assert len(table) > 1, 'table must have at least a header and a row'
            assert len(table[0]) > 0, 'table must have at least one column'
            self.onedim = len(table) == 2
            if self.onedim:
                self.top = [pyToZ3(e) for e in table[0]] # convert Python bool, int, float ...
                self.body = [pyToZ3(e) for e in table[1]] # ... constants to Z3 types
            else:
                self.top = [pyToZ3(e) for e in table[0]]
                self.left = [pyToZ3(row[0]) for row in table[1:]]
                self.body = [[pyToZ3(e) for e in row[1:]] for row in table[1:]]
        else: # table must be empty, top, left, body must be valid
            assert len(table) == 0, 'ambiguous table specification'
            assert top != None and len(top) > 0, 'top header must have at least one column'
            self.onedim = left == None
            if not self.onedim:
                self.top = [pyToZ3(e) for e in top]
                self.left = [pyToZ3(e) for e in left]
                self.body = [[pyToZ3(e) for e in row] for row in body]
        assert all(isinstance(h, z3.BoolRef) for h in self.top), 'top header element not Boolean'
        if self.onedim:
            types = {z3.BoolRef if isinstance(e, z3.BoolRef) else
                     z3.ArithRef if isinstance(e, z3.ArithRef) else
                     e.typ if isinstance(e, (Table, VectorTable)) else
                     z3.BoolRef if isinstance(e, str) and isinstance(eval(e), z3.BoolRef) else
                     z3.ArithRef if isinstance(e, str) and isinstance(eval(e), z3.ArithRef) else
                     eval(e).typ if isinstance(e, str) and isinstance(eval(e), (Table, VectorTable)) else type(e)
                         for e in self.body}
            assert len(types) == 1, 'body elements not of the same type: ' + str(types)
            assert len(table[0]) == len(table[1]), 'table size mismatch'
            self.total, self.totalcounterexample = checktotal(self.top)
            self.disjoint, self.disjointcounterexample = checkdisjoint(self.top)
            self.normal = all('ʹ' not in h.mathml() for h in self.top)
        else:
            assert all(isinstance(h, z3.BoolRef) for h in self.left), 'left header element not Boolean'
            types = {z3.BoolRef if isinstance(e, z3.BoolRef) else
                     z3.ArithRef if isinstance(e, z3.ArithRef) else
                     e.typ if isinstance(e, (Table, VectorTable)) else
                     z3.BoolRef if isinstance(e, str) and isinstance(eval(e), z3.BoolRef) else
                     z3.ArithRef if isinstance(e, str) and isinstance(eval(e), z3.ArithRef) else
                     eval(e).typ if isinstance(e, str) and isinstance(eval(e), (Table, VectorTable)) else type(e)
                         for row in self.body for e in row}
            assert len(types) == 1, 'body elements not of the same type: ' + str(types)
            assert all(len(row) == len(self.top) for row in self.body), 'table size mismatch'
            self.lefttotal, self.lefttotalcounterexample = checktotal(self.left)
            self.toptotal, self.toptotalcounterexample = checktotal(self.top)
            self.leftdisjoint, self.leftdisjointcounterexample = checkdisjoint(self.left)
            self.topdisjoint, self.topdisjointcounterexample = checkdisjoint(self.top)
            self.total, self.disjoint = self.lefttotal and self.toptotal, self.leftdisjoint and self.topdisjoint
            self.normal = all('ʹ' not in h.mathml() for h in self.top + self.left)
        self.typ = types.pop()
    def mathml(self, precedence = 0) -> str:
        if self.onedim:
            toptotal = 'border-bottom:double black;' if self.total else \
                'border-bottom:solid black;' if self.total == False else 'border-bottom:dotted grey;'
            topdisjoint = 'border-left:solid black;' if self.disjoint else \
                'border-left:thin solid black;' if self.disjoint == False else 'border-left:dotted grey;'
            head = '<mtr>\n' + \
                '  <mtd style="text-align:center;' + toptotal + '">' + \
                     ('</mtd>\n  <mtd style="text-align:center;' + toptotal + topdisjoint + '">').join(h if isinstance(h, str) else h.mathml() for h in self.top) + \
                   '</mtd>\n' + \
                '</mtr>\n'
            tail = '<mtr>\n' + \
                '  <mtd style="text-align:center">' + \
                     ('</mtd>\n  <mtd style="text-align:center;' + topdisjoint + '">').join(b if isinstance(b, str) else b.mathml() for b in self.body) + \
                   '</mtd>\n' + \
                '</mtr>\n'
        else:
            toptotal = 'border-bottom:double black;' if self.toptotal else \
                'border-bottom:solid black;' if self.toptotal == False else 'border-bottom:dotted grey;'
            topdisjoint = 'border-left:solid black;' if self.topdisjoint else \
            'border-left:thin solid black;' if self.topdisjoint == False else 'border-left:dotted grey;'
            lefttotal = 'border-right:thick double black;' if self.lefttotal else \
                'border-right:solid black;' if self.lefttotal == False else 'border-right:dotted grey;'
            leftdisjoint = 'border-top:solid black;' if self.leftdisjoint else \
                'border-top:thin solid black;' if self.leftdisjoint == False else 'border-top:dotted grey;'
            head = '<mtr>\n' + \
                '  <mtd style="' + toptotal + lefttotal + '">\n' + \
                '  </mtd><mtd style="' + toptotal + '">' + \
                     self.top[0].mathml() + '</mtd>\n  <mtd style="' + toptotal + topdisjoint + '">' + \
                     ('</mtd>\n  <mtd style="' + toptotal + topdisjoint + '">').join(h if isinstance(h, str) else h.mathml() for h in self.top[1:]) + \
                   '</mtd>\n' + \
                '</mtr>\n' # first column of header is empty
            tail = '<mtr>\n' + \
                '  <mtd style="' + lefttotal + '">' + self.left[0].mathml() + '</mtd>\n' + \
                '  <mtd style="' + '">' + \
                        ('</mtd>\n  <mtd style="' + topdisjoint + '">').join(b if isinstance(b, str) else b.mathml() for b in self.body[0]) + \
                    '</mtd>\n' + \
                '</mtr>\n'
            for r, row in enumerate(self.body[1:]):
                tail += '<mtr>\n' + \
                    '  <mtd style="' + lefttotal + leftdisjoint + '">' + self.left[r + 1].mathml() + '</mtd>\n' + \
                    '  <mtd style="' + leftdisjoint + '">' + \
                         ('</mtd>\n  <mtd style="' + topdisjoint + leftdisjoint + '">').join(b if isinstance(b, str) else b.mathml() for b in row) + \
                       '</mtd>\n' + \
                    '</mtr>\n'
            # tail = '<tr>\n' + \
            #     '  <td style="text-align:center;' + lefttotal + '">' + str(self.left[0]) + '</td>\n' + \
            #     '  <td style="text-align:center;">' + \
            #          ('</td>\n  <td style="text-align:center;' + leftdisjoint + '">').join(map(str, self.body)) + \
            #        '</td>\n' + \
            #     '</tr>\n'
        return '<mspace width=".1em"/><mtable style="text-align:center;">\n' + head + tail + '</mtable><mspace width=".1em"/>'

In [ ]:
def flattenheader(h: z3.BoolRef | list | tuple) -> tuple:
    r = ()
    for e in h: # elements of header
        if isinstance(e, (list, tuple)):
            assert len(e) == 2, 'nested header element must have two rows'
            assert isinstance(e[0], z3.BoolRef), 'first element of a nested header not Boolean'
            r += tuple(e[0] & c for c in flattenheader(e[1]))
        elif isinstance(e, z3.BoolRef): r += (e,)
        else: assert False, 'header element not Boolean or nested'
    return r

In [ ]:
if TEST: assert flattenheader((a, (~a, (b, ~b)))) == (a, ~a & b, ~a & ~b)

In [ ]:
class VectorTable(SpecExpr):
    def __init__(self, *table, top = None, left = None, body = None):
        if body == None: # table must be well-formed
            assert len(table) > 1, 'table must have at least a header and a row'
            assert len(table[0]) > 0, 'table must have at least one column'
            assert top == None and left == None, 'inconsistent table specification'
            self.disptop = pyToZ3(table[0])
            self.left = tuple(pyToZ3(row[0]) for row in table[1:])
            self.body = tuple(pyToZ3(row[1:]) for row in table[1:])
        else: # table must be empty, top, left, body must be valid
            assert len(table) == 0, 'ambiguous table specification'
            assert top != None and len(top) > 0, 'top header must have at least one column'
            # self.top = [pyToZ3(e) for e in top]
            # self.left = [pyToZ3(e) for e in left]
            # self.body = [[pyToZ3(e) for e in row] for row in body]
            self.disptop, self.left, self.body = pyToZ3(top), pyToZ3(left), pyToZ3(body)
        # assert all(isinstance(h, z3.BoolRef) for h in self.top), 'top header element not Boolean'
        self.top = flattenheader(self.disptop) # checks if header is Boolean
        for row in self.body:
            types = {z3.BoolRef if isinstance(e, z3.BoolRef) else
                     z3.ArithRef if isinstance(e, z3.ArithRef) else
                     e.typ if isinstance(e, (Table, VectorTable)) else
                     z3.BoolRef if isinstance(e, str) and isinstance(eval(e), z3.BoolRef) else
                     z3.ArithRef if isinstance(e, str) and isinstance(eval(e), z3.ArithRef) else
                     eval(e).typ if isinstance(e, str) and isinstance(eval(e), (Table, VectorTable)) else type(e)
                         for e in row}
            assert len(types) == 1, 'body elements not of the same type: ' + str(types)
        assert all(len(row) == len(self.top) for row in self.body), 'table size mismatch'
        self.toptotal, self.toptotalcounterexample = checktotal(self.top)
        self.topdisjoint, self.topdisjointcounterexample = checkdisjoint(self.top)
        self.total, self.disjoint = self.toptotal, self.topdisjoint
        self.normal = all('ʹ' not in h.mathml() for h in self.top + self.left)
        self.typ = z3.z3.BoolRef
    def mathml(self, precedence = 0) -> str:
        toptotal = 'border-bottom:double black;' if self.toptotal else \
            'border-bottom:solid black;' if self.toptotal == False else 'border-bottom:dotted grey;'
        topdisjoint = 'border-left:solid black;' if self.topdisjoint else \
        'border-left:thin solid black;' if self.topdisjoint == False else 'border-left:dotted grey;'
        lefttotal = 'border-right:solid black;'
        leftdisjoint = 'border-top:solid black;'
        head = topHeadMathML(self.disptop, lefttotal, toptotal, topdisjoint)
        # head = ('<mtr>\n' + # head row start: first column of header (in next line) is empty
        #     '  <mtd style="' + toptotal + lefttotal + '"></mtd>\n' + # bottom border (toptotal) and right border (lefttotal) set
        #     '  <mtd style="' + toptotal + '">' +  # second header element (first above body):  bottom border (toptotal) and right border (topdisjoint) set
        #          self.top[0].mathml() + '</mtd>\n  <mtd style="' + toptotal + topdisjoint + '">' + # 
        #          ('</mtd>\n  <mtd style="' + toptotal + topdisjoint + '">').join(h.mathml() for h in self.top[1:]) +
        #        '</mtd>\n' +
        #     '</mtr>\n')
        tail = '<mtr>\n' + \
            '  <mtd style="' + lefttotal + '">' + self.left[0].mathml() + ' = </mtd>\n' + \
            '  <mtd style="' + '">' + \
                    ('</mtd>\n  <mtd style="' + topdisjoint + '">').join(b if isinstance(b, str) else b.mathml() for b in self.body[0]) + \
                '</mtd>\n' + \
            '</mtr>\n'
        for r, row in enumerate(self.body[1:]):
            tail += '<mtr>\n' + \
                '  <mtd style="' + lefttotal + leftdisjoint + '">' + self.left[r + 1].mathml() + ' = </mtd>\n' + \
                '  <mtd style="' + leftdisjoint + '">' + \
                     ('</mtd>\n  <mtd style="text-align:center;' + topdisjoint + leftdisjoint + '">').join(b if isinstance(b, str) else b.mathml() for b in row) + \
                   '</mtd>\n' + \
                '</mtr>\n'
        return '<mspace width=".1em"/><mtable style="text-align:center;">\n' + head + tail + '</mtable><mspace width=".1em"/>'

def topHeadMathML(top: list | tuple, lefttotal: str, toptotal: str, topdisjoint: str) -> str: # returns string of <mtr>'s with enclosed <mtd>'s
    # lefttotal: right border of leftmost empty shared header element
    # toptotal: bottom border of all header elements
    # topdisjoint: left border from the third header element onwards
    depth = lambda L: max(map(depth, L[1])) + 1 if isinstance(L, tuple) else 1 # if L is tuple, it must be a pair (any, tuple)
    width = lambda L: sum(map(width, L[1])) if isinstance(L, tuple) else 1 # if L is tuple, it must a pair (any, tuple)
    nesting = max(map(depth, top)) # maximal nesting of subheaders, needed to determine the rowspan of header elements
    rowspan = '' if nesting == 1 else ' rowspan=' + str(nesting) # height of top left empty shared header element
    rows = '<mtr><mtd' + rowspan + ' style="' + toptotal + lefttotal + '"></mtd>' # height, bottom border (toptotal), and right border (lefttotal)
    cur, nxt, cursep, nxtsep = top, (), ('',) + (len(top) - 1) * (topdisjoint,), () # current and next header row, current and next separators, of matching size
    for n in range(nesting): # for each header row
        if n != 0: rows += '<mtr>' # the topmost header has <mtr> already from the leftmost empty share header element
        for h, s in zip(cur, cursep): # for each header row and corresponding separator
            if isinstance(h, tuple):
                colspan = ' columnspan=' + str(width(h))
                rows += '<mtd' + colspan + ' style="border-bottom:thin solid black;'+ s + '">' + h[0].mathml() + '</mtd>'
                nxt += h[1] # subheaders are added to the next row
                nxtsep += (s,) + (len(nxt) - 1) * ('border-left:thin solid black;',) # first seperator is inherited, remaining ones are thin
            else:
                rowspan = '' if nesting - n == 1 else ' rowspan=' + str(nesting - n)
                rows += '<mtd' + rowspan + ' style="' + toptotal +  s + '">' + h.mathml() + '</mtd>'
        rows += '</mtr>\n'
        cur, nxt, cursep, nxtsep = nxt, (), nxtsep, ()
    return rows

In [ ]:
# v = VectorTable(((a, (b, ~b)), (~a, ((b, (c, ~c)), ~b))), (x, 3, 5, 7, 8, 9), (x + y, x - y, 1, 3, 8, 5)); v

In [ ]:
# Nested MathML tables cannot be used to display nested headers: the vertical borders of the header do not align with those of the body.
# h = """
# <math style="border-left:2em solid white;font-family: sans-serif;"><mspace width=".1em"/>
# <mtable style="text-align:center;">
# <mtr>
#   <mtd style="border-bottom:double black;border-right:solid black;"></mtd>
#   <mtd style="border-bottom:double black;">a</mtd>
#   <mtd columnspan=2 style="border-bottom:double black;border-left:solid black;">
#     <mtable>
#       <mtr style="text-align:center;">
#         <mtd columnspan=2 style="border-bottom:1px solid black;">&not;a</mtd>
#       </mtr>
#       <mtr>
#         <mtd style="text-align:center;border-right:1px solid black">b</mtd>
#         <mtd style="text-align:center;">&not;b</mtd>
#       </mtr>
#   </mtable>
#   </mtd>
# </mtr>
# <mtr>
#   <mtd style="border-right:solid black;">x = </mtd>
#   <mtd style="">7</mtd>
#   <mtd style="border-left:solid black;">8</mtd>
#   <mtd style="border-left:solid black;">9</mtd>
# </mtr>
# <mtr>
#   <mtd style="border-right:solid black;border-top:solid black;">x + y = </mtd>
#   <mtd style="border-top:solid black;">x - y</mtd>
#   <mtd style="border-left:solid black;border-top:solid black;">8</mtd>
#   <mtd style="border-left:solid black;border-top:solid black;">5</mtd>
# </mtr>
# </mtable><mspace width=".1em"/></math>
# """
# HTML(h)

In [ ]:
# w = VectorTable(top = (a, b), left = (a, b), body = ((~a, a), ('v', b))); w

In [ ]:
#flatten(v)

The Python operator `>>` is added for implication. However, `⇒` binds tighter, not weaker, than `∧` and `∨` (it is also used for tabular expressions, where it is more useful):

In [ ]:
# def impliesExpr(arg0, arg1):
#     return ImpliesExpr(arg0, arg1)
# setattr(z3.ExprRef, '__rshift__', impliesExpr)

# def consExpr(arg0, arg1):
#     return ConsExpr(arg0, arg1)
# setattr(z3.ExprRef, '__lshift__', consExpr)

setattr(z3.ExprRef, '__rshift__', Implies)

#### Constructing Tables

Method `_repr_html_` of Z3 expressions uses HTML `<sup>` for superscripts. If a Z3 expression appears in a table, the superscript is not displayed properly, as tables use MathML, which has `<msup>` and ignores `<sup>`. The method `mathml` is added to the existing Z3 expressions: it takes the HTML representation of a Z3 expression and converts it to MathML.

In [ ]:
def htlm_to_mathml(e: z3.ExprRef):
    return e._repr_html_() \
            .replace('<sup>', '<msup><mn></mn>').replace('</sup>', '</msup>') \
            .replace('<sub>', '<msub><mn></mn>').replace('</sub>', '</msub>') \
            .replace('&exist;', '<mo form="prefix" rspace=".1em" largeop="true">&exist; </mo>') \
            .replace('&forall;', '<mo form="prefix" rspace=".1em" largeop="true">&forall; </mo>')

setattr(z3.z3.ExprRef, 'mathml', htlm_to_mathml)

This pretty-prints Z3 expressions in tables.

#### Free and Primed Variables

In [ ]:
def freevars(expr: z3.ExprRef, bound = set()):
    free = set()
    if z3.is_const(expr): # Case constants (uninterpreted symbols)
        decl = expr.decl()
        if decl.kind() == z3.Z3_OP_UNINTERPRETED and expr not in bound:
            free.add(expr)
    elif z3.is_app(expr): # Case function applications or arrays
        decl = expr.decl()
        # If it's an uninterpreted function symbol (like f)
        if decl.kind() == z3.Z3_OP_UNINTERPRETED and expr.num_args() > 0:
            free.add(decl) # Add the function symbol itself
        for ch in expr.children(): # Recurse into all arguments
            free |= freevars(ch, bound)
    elif z3.is_quantifier(expr): # Case quantifiers
        new_bound = bound.copy()
        for i in range(expr.num_vars()):
            v = z3.Const(expr.var_name(i), expr.var_sort(i))
            new_bound.add(v)
        free |= freevars(expr.body(), new_bound)
    return free
# def freevars(expr, bound = set()):
#     free = set()
#     if z3.is_const(expr) and expr.decl().kind() == z3.Z3_OP_UNINTERPRETED:
#         # A constant symbol that is not bound
#         if expr not in bound: free.add(expr)
#     elif z3.is_quantifier(expr):
#         # When encountering a quantifier, add its bound vars to the "bound" set
#         new_bound = bound.copy()
#         for i in range(expr.num_vars()):
#             v = z3.Const(expr.var_name(i), expr.var_sort(i))
#             new_bound.add(v)
#         # Recurse into the quantified body
#         free |= freevars(expr.body(), new_bound)
#     else:
#         # Recurse into children for composite expressions
#         for ch in expr.children():
#             free |= freevars(ch, bound)
#     return free

In [ ]:
if TEST: assert freevars(Exists(xʹ, xʹ > yʹ)) == {yʹ}

Example with Z3 functions and Z3 arrays:

In [ ]:
if TEST:
    A = z3.Array('A', z3.IntSort(), z3.IntSort())
    f = z3.Function('f', z3.IntSort(), z3.IntSort(), z3.IntSort())
    assert freevars(ForAll(x, f(y, A[z]) > z3.Store(A, x, y)[z])) == {A, f, y, z}

The function `primedvars` returns a list with the primed free variables in a tabular expression.

In [ ]:
def primedvars(t: Expr):
    match t:
        case Table():
            vt = {v for h in t.top for v in freevars(h) if str(v).endswith('ʹ')}
            if t.onedim:
                vb = {v for h in t.body for v in freevars(h) if str(v).endswith('ʹ')}
                return vt | vb
            else:
                vl = {v for h in t.left for v in freevars(h) if str(v).endswith('ʹ')}
                vb = {v for r in t.body for c in r for v in freevars(c) if str(v).endswith('ʹ')}
                return vt | vl | vb
        case VectorTable():
            vt = {v for h in t.top for v in freevars(h) if str(v).endswith('ʹ')}
            vl = {v for h in t.left for v in freevars(h) if str(v).endswith('ʹ')}
            vb = {v for r in t.body for c in r for v in freevars(c) if str(v).endswith('ʹ')}
            return vt | vl | vb
        case NotExpr():
            return primedvars(t.arg)
        case AndExpr() | OrExpr() | ImpliesExpr() | ConsExpr() | EqExpr() | NeExpr() | Possible() | Necessary():
            return primedvars(t.arg0) | primedvars(t.arg1)
        case Dom() | Correct(): return set()
        case z3.ExprRef(): return {v for v in freevars(t) if str(v).endswith('ʹ')}
        case _: assert False, 'unexpected case'

In [ ]:
if TEST:
    t = Table((a, ~aʹ), (x + y >= 3, True))
    assert primedvars(t) == {aʹ}

In [ ]:
if TEST:
    u = Table((a, ~aʹ), (x + y >= 3, "t"))
    u = Table((a, ~aʹ), (x + y >= 3, t))
# u

In [ ]:
if TEST:
    t = Table((a, ~a), (xʹ + y >= 3, xʹ == x))
    assert primedvars(t) == {xʹ}

In [ ]:
if TEST:
    assert primedvars(Exists(xʹ, xʹ > yʹ)) == {yʹ}

In [ ]:
def primedpairs(vs):
    return [(globals()[str(v).removesuffix('ʹ')], v) for v in vs]

#### Substitution

Function `substitute(t, s)` takes a (tabular or flat) expression `t` and a list of expression pairs, and returns `t` with all expressions simultaneously substituted.

In [ ]:
def substitute(t: Expr, s: list[tuple[z3.ExprRef, z3.ExprRef]]) -> Expr:
    match t:
        case Table():
            top = substitute(t.top, s)
            body = substitute(t.body, s)
            if t.onedim:
                return Table(top, body)
            else:
                left = substitute(t.left, s)
                return Table(top = top, left = left, body = body)
        case VectorTable():
            top = substitute(t.top, s)
            body = substitute(t.body, s)
            left = substitute(t.left, s)
            return VectorTable(top = top, left = left, body = body)
        case NotExpr(): return NotExpr(substitute(t.arg, s))
        case AndExpr(): return AndExpr(substitute(t.arg0, s), substitute(t.arg1, s))
        case OrExpr(): return OrExpr(substitute(t.arg0, s), substitute(t.arg1, s))
        case ImpliesExpr(): return ImpliesExpr(substitute(t.arg0, s), substitute(t.arg1, s))
        case ConsExpr(): return ConsExpr(substitute(t.arg0, s), substitute(t.arg1, s))
        case EqExpr(): return EqExpr(substitute(t.arg0, s), substitute(t.arg1, s))
        case NeExpr(): return NeExprsubstitute(t.arg, s)
        case Possible(): return Possible(substitute(t.arg0, s), substitute(t.arg1, s))
        case Necessary(): return Necessary(substitute(t.arg0, s), substitute(t.arg1, s))
        case tuple(): return tuple(substitute(f, s) for f in t)
        case list(): return [substitute(f, s) for f in t]
        case z3.ExprRef(): return z3.substitute(t, s)
        case int(): return t
        case bool(): return t
        case _: assert False, 'unexpected case'

In [ ]:
if TEST: t = Table((a, ~a), (x + y >= 3, True)); t

In [ ]:
if TEST: u = Table((a, ~a), (x + y >= 3, xʹ == x)); u

In [ ]:
if TEST: substitute(Necessary(t, u), [(x, xʹ)])

#### Flattening Tabular Expressions

Flattening tables removes the one- or two-dimensional structure of tables and eliminates the `∆`, `⟨_⟩_`, `[_]_`, and `{_}_{_}` operators. Flattening does not further simplify the tables.

In [ ]:
def flatten(t: Expr) -> Expr:
    match t:
        case Table():
            return Or(map(And, map(flatten, t.top), map(flatten, t.body))) if t.onedim else \
                Or(flatten(l) & tb for row in t.body for l in t.left for tb in map(And, map(flatten, t.top), map(flatten, row)))
        case VectorTable():
            return Or(t.top[i] & And(t.left[j] == t.body[j][i] for j in range(len(t.left))) for i in range(len(t.top)))
            return Or(t.top[i] & And(t.left[j] == t.body[i, j] for j in range(len(t.left))) for i in range(len(t.top)))
        case NotExpr(): return ~flatten(t.arg)
        case AndExpr(): return flatten(t.arg0) & flatten(t.arg1)
        case OrExpr(): return flatten(t.arg0) | flatten(t.arg1)
        case ImpliesExpr(): return flatten(t.arg0) >> flatten(t.arg1)
        case ConsExpr(): return flatten(t.arg1) >> flatten(t.arg0)
        case EqExpr(): return flatten(t.arg0) == flatten(t.arg1)
        case NeExpr(): return flatten(t.arg0) != flatten(t.arg1)
        case Dom(): return Exists(list(primedvars(t.arg)), flatten(t.arg))
        case Possible():
            arg0, arg1 = flatten(t.arg0), flatten(t.arg1)
            pv = primedvars(arg0)
            return Exists(list(pv), arg0 & substitute(arg1, primedpairs(pv)))
        case Necessary():
            arg0, arg1 = flatten(t.arg0), flatten(t.arg1)
            pv = primedvars(arg0)
            return ForAll(list(pv), arg0 >> substitute(arg1, primedpairs(pv)))
        case Correct():
            return flatten(t.arg0) >> flatten(Necessary(t.arg1, t.arg2))
        case str(): return flatten(eval(t))
        case _: return t

#### Timed Specifications

Time is captured by the integer variable `time`. A tick event increases `time` by `1` and executes any operations that are scheduled at the new time. Only one operation can be scheduled at a certain time. If the operation cannot be executed, its primed variables remain unchanged.

In [ ]:
Int('time')

def Timer(timeout, handler):
    from itertools import starmap
    nochange = z3.And(starmap(lambda a, b: b == a, primedpairs(primedvars(handler))))
    timeoutprimed = substitute(timeout, (time, timeʹ)) # timeout with time in timeout expression primed
    return (time >= 0) & (timeʹ == time + 1) & ((timeoutprimed & substitute(flatten(handler), (time, timeʹ))) | (~timeoutprimed & nochange))

The following displays `timeʹ = time + 1 ∧ (timeʹ = 5 ∧ xʹ + yʹ = z ∨ ¬(timeʹ = 5) ∧ y = yʹ ∧ x = xʹ)`.

In [ ]:
if TEST: t = Timer(time == 5, xʹ + yʹ == z); display(t)

#### Validity of Tabular Expressions

Once a table expression is flattened into a Z3 expression, it can be checked for validity:

In [ ]:
def valid(t):
    s = z3.Solver(); s.add(~flatten(t))
    return s.check() == z3.unsat

In [ ]:
# def valid(t: Expr) -> bool:
#     match t:
#         case Dom() if isinstance(t.arg, Table) and t.arg.normal:
#             if t.arg.onedim:
#                 body = [Domain(c) for c in t.arg.body]
#                 return valid(Table(t.arg.top, body))
#             else:
#                 body = [[Domain(c) for c in r] for r in t.arg.body]
#                 return valid(Table(top = t.arg.top, left = t.arg.left, body = body))
#         case Possible() if isinstance(t.arg0, Table) and t.arg0.normal:
#             if t.arg0.onedim:
#                 body = [Possible(c, t.arg1) for c in t.arg0.body]
#                 return valid(Table(t.arg0.top, body))
#             else:
#                 body = [[Possible(c, t.arg1) for c in r] for r in t.arg0.body]
#                 return valid(Table(top = t.arg0.top, left = t.arg0.left, body = body))
#         case Necessary() if isinstance(t.arg0, Table) and t.arg0.normal and t.arg0.disjoint:
#             if t.arg0.onedim:
#                 body = [Necessary(c, t.arg1) for c in t.arg0.body]
#                 return valid(Table(t.arg0.top, body))
#             else:
#                 body = [[Necessary(c, t.arg1) for c in r] for r in t.arg0.body]
#                 return valid(Table(top = t.arg0.top, left = t.arg0.left, body = body))
#         case Correct() if isinstance(t.arg1, Table) and t.arg1.normal and \
#                           t.arg1.total and t.arg1.disjoint:
#             if t.arg2.onedim:
#                 body = [Necessary(c, t.arg2) for c in t.arg2.body]
#                 return valid(Table(t.arg2.top, body))
#             else:
#                 body = [[Necessary(c, t.arg3) for c in r] for r in t.arg2.body]
#                 return valid(Table(top = t.arg1.top, left = t.arg1.left, body = body))
#         #case Necessary() if isinstance(t.arg0, Table) and t.arg0.normal:
#         # case Table():
#         #     if t.onedim:
                
#     s = z3.Solver(); s.add(flatten(t))
#     return s.check() == z3.sat

#### Simplifying Tables – Experimental: Ignore

In [ ]:
def simplify(t: Table):
    def cons(e, l): return [e] + l
    def headerbody(t):
        return (list(map(simplify, t.top)),
            (list(map(simplify, t.body)) if t.onedim else \
            list(map(simplify, row) for row in map(cons, t.left, t.body))))
    def applyheader(f, l):
        return list(map(f, l))
    def applybody(f, t):
        return list(map(f, t.body)) if t.onedim else list(map(f, row) for row in t.body)
    match t:
        case Table():
            return Table(*headerbody(t))
        case NotExpr():
            return NotTable(*headerbody(t)) if t.total and t.disjoint else t
        case AndExpr():
            return AndTable(*headerbody(t.arg0), *headerbody(t.arg1))
        case OrExpr():
            return OrTable(*headerbody(t.arg0), *headerbody(t.arg1))
        case ImpliesExpr():
            return ImpliesTable(*headerbody(t.arg0), *headerbody(t.arg1))
        case EqExpr():
            return Table(*headerbody(t.arg0)) == Table(*headerbody(t.arg1))
        case Dom():
            return Table(*headerbody(t))
        case Possible():
            return Possible(flatten(t.arg0), flatten(t.arg1))
        case Necessary():
            return Necessary(flatten(t.arg0), flatten(t.arg1))
        case _:
            return z3.simplify(t)

In [ ]:
#simplify(t)

In [ ]:
#simplify(flatten(t))

In [ ]:
#z3.simplify((x == y + 1) >> (x + y + 2 * x == 3 *y + 3))

In [ ]:
#g = z3.Goal(); g.add(x == y + 1); g.add(x + y + 2 * x == 3 *y + 3)

In [ ]:
#g = z3.Solver(); g.add(x == y + 1); g.add(x + y + 2 * x == 3 *y + 3); g.check()

In [ ]:
# t = z3.Tactic('simplify'); t(g)

In [ ]:
# def pp_bool(a: z3.z3.BoolRef):
#     def a0(): return a.arg(0)._repr_html_()
#     def a1(): return a.arg(1)._repr_html_()
#     # print(a.decl())
#     # print(type(a.decl()))
#     # op = a.decl().name()
#     # print(op)
#     if type(a) == z3.QuantifierRef:
#         return '∀' if a.is_forall() else \
#             '∃ ' + a.var_name(0) + ' ∙ ' + a.body().mathml() if a.is_exists() else a
#     else:
#         op = a.decl().name()
#         #op = a.kind()
#         return '¬ ' + a0() if op == 265 or op == 'not' else \
#            a0() + ' ∧ ' + a1() if op == 'and' else \
#            a0() + ' ∨ ' + a1() if op == 'or' else \
#            a0() + ' = ' + a1() if op == '=' or op == 258 else \
#            a0() + ' < ' + a1() if op == '<' else \
#            a0() + ' > ' + a1() if op == '>' else \
#            a0() + ' ≤ ' + a1() if op == '<=' else \
#            a0() + ' ≥ ' + a1() if op == '>=' else str(a)

# #setattr(z3.z3.BoolRef, '_repr_html_', pp_bool)
# #setattr(z3.z3.BoolRef, 'mathml', pp_bool)

In [ ]:
# def pp_arith(a: z3.z3.ArithRef):
#     def a0(): return a.arg(0)._repr_html_()
#     def a1(): return a.arg(1)._repr_html_()
#     print('arith', a, type(a))
#     global aa; aa = a
#     if a.is_int() or a.is_real():
#         print(type(a))
#         return a
#     else:
#         op = a.decl().name()
#         return a0() + ' ' + op + ' '+ a1() if op in '+-' else \
#                a0() + '<msup>' + a1() + '</msup>' if op == '^' else \
#                a.as_string() if op in ('Int', 'Bool', 'Real') else str(a)

# #setattr(z3.z3.ArithRef, '_repr_html_', pp_arith)
# #setattr(z3.z3.ArithRef, 'mathml', pp_arith)

In [ ]:
# Or(And(l, tb) for row in [[aʹ, bʹ], [cʹ, dʹ]] for l in [a, b] for tb in map(And, [c, d], row))

In [ ]:
# aʹ == a

In [ ]:
# t & t

In [ ]:
# u = t >> (x > 0); u

In [ ]:
# flatten(u)

In [ ]:
# Dom(t)

In [ ]:
# Possible(t, x > 0)

In [ ]:
# Possible(x > y, x > 0)

In [ ]:
# Necessary(t, x > 0)